In [ ]:
import os

os.chdir("../")

In [ ]:
import torch
import fitz
import io
from PIL import Image
from transformers import AutoModelForTokenClassification, AutoProcessor
from collections import Counter

In [ ]:
pdf_path = "path/to/your_invoice.pdf"

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    "app/machine_learning/separated/layoutlmv3-invoice-model-separated-training_args/checkpoint-300"
)

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

In [ ]:
def get_page_object(uploaded_file, dpi=300):
    doc = fitz.open(uploaded_file)
    page = doc[0]
    page_area = page.rect.width * page.rect.height
    adaptive_dpi = min(dpi, max(150, int(300000 / page_area * 72)))
    mat = fitz.Matrix(adaptive_dpi / 72, adaptive_dpi / 72)
    pix = page.get_pixmap(matrix=mat)
    img_data = pix.tobytes("png")
    image = Image.open(io.BytesIO(img_data))

    return image

In [ ]:
def aggregate_page_data(uploaded_file):
    doc = fitz.open(uploaded_file)
    total_pages = len(doc)

    file_data = {"csv_entry": {}, "pages": {}}

    for i in range(total_pages):
        page = doc.load_page(i)

        words_data = page.get_text("words")

        file_data["pages"][i + 1] = {
            "words_data": words_data,
            "annotations": {},
            "pdf_width": page.rect.width,
            "pdf_height": page.rect.height,
        }

    doc.close()
    return file_data

In [ ]:
image = get_page_object(pdf_path)
file_data = aggregate_page_data(pdf_path)

In [ ]:
page_data = file_data["pages"][1]
raw_words_data = page_data["words_data"]
pdf_width = page_data["pdf_width"]
pdf_height = page_data["pdf_height"]

In [ ]:
words = []
bboxes = []
for i, entry in enumerate(raw_words_data):
    words.append(raw_words_data[i][4])
    bbox = [
        raw_words_data[i][0],
        raw_words_data[i][1],
        raw_words_data[i][2],
        raw_words_data[i][3],
    ]
    bboxes.append(bbox)

In [ ]:
def normalize_box(bbox, width, height):
    return [
        int(1000 * (bbox[0] / width)),
        int(1000 * (bbox[1] / height)),
        int(1000 * (bbox[2] / width)),
        int(1000 * (bbox[3] / height)),
    ]


def normalize_bboxes(bboxes, width, height):
    new_boxes = []
    for i, box in enumerate(bboxes):
        transformed = normalize_box(box, width, height)
        new_boxes.append(transformed)
    return new_boxes

In [ ]:
normalized_bboxes = normalize_bboxes(bboxes, pdf_width, pdf_height)

In [ ]:
encoding = processor(
    image,
    words,
    boxes=normalized_bboxes,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
    max_length=512,
)

In [ ]:
with torch.no_grad():
    outputs = model(**encoding)

logits = outputs.logits
predictions = logits.argmax(-1).squeeze().tolist()

id2label = model.config.id2label

predicted_labels = [id2label[p] for p in predictions]

word_ids = encoding.word_ids()

In [ ]:
import torch.nn.functional as F

probs = F.softmax(logits, dim=-1)
confidences, _ = probs.max(dim=-1)
confidences = confidences.squeeze().tolist()

final_confidences = []

for i, word_id in enumerate(word_ids):
    if word_id is None:
        continue
    if i == 0 or word_id != word_ids[i - 1]:
        final_confidences.append(confidences[i])

In [ ]:
final_labels = []
final_boxes = []
final_words = []

for i, word_id in enumerate(word_ids):
    if word_id is None:
        continue  # skip special/padding tokens

    # If this token is the first subtoken of a word
    if i == 0 or word_id != word_ids[i - 1]:
        final_labels.append(id2label[predictions[i]])
        final_boxes.append(encoding.bbox.squeeze().tolist()[i])
        final_words.append(words[word_id])

In [ ]:
import pandas as pd

In [ ]:
final_df = pd.DataFrame(
    {
        "label": final_labels,
        "word": final_words,
        "confidence": final_confidences,
        "boxes": final_boxes,
    }
)

In [ ]:
result = final_df.loc[final_df.groupby("label")["confidence"].idxmax()]

In [ ]:
COLUMNS = [
    "file_name",
    "invoice_number",
    "invoice_date",
    "issuer_name",
    "issuer_office",
    "receiver_name",
    "receiver_office",
    "line_items",
    "line_item_amounts",
    "vat",
    "total_amount",
    "due_date",
    "period_start",
    "period_end",
    "custom_field_1",
    "custom_field_2",
]

In [ ]:
for i, col in enumerate(COLUMNS):
    for index, row in result.iterrows():
        if row[0].endswith(col.upper()) and (
            row[0].startswith("B") or row[0].startswith("S")
        ):
            print(
                row[0],
            )

In [ ]:
entries = []
boxes = []
for i, col in enumerate(COLUMNS):
    col_entries = ""
    col_boxes = []
    for index, row in result.iterrows():
        if row[0].endswith(col.upper()) and (
            row[0].startswith("B") or row[0].startswith("S")
        ):
            col_entries = row[1]
            col_boxes.append(row[3])
        if row[0].endswith(col.upper()) and (row[0].startswith("I")):
            col_entries += " " + row[1]
            col_boxes.append(row[3])
    entries.append(col_entries)
    boxes.append(col_boxes)